# llmevalkit v2.0 - Quality Evaluation + Compliance Testing

**15 quality metrics + 6 compliance metrics in one library.**

- Part A: Quality evaluation (local metrics, free, no API)
- Part B: Compliance testing (PII, HIPAA, GDPR, DPDP, EU AI Act)
- Part C: Combined quality + compliance
- Part D: LLM-as-judge metrics (needs API key)

GitHub: https://github.com/VK-Ant/llmevalkit

Author: Venkatkumar Rajan (@VK_Venkatkumar)

In [1]:
!pip install llmevalkit -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.2/52.2 kB 2.6 MB/s eta 0:00:00


## Part A: Quality Evaluation (free, no API)

In [2]:
from llmevalkit import Evaluator

evaluator = Evaluator(provider="none", preset="math")
result = evaluator.evaluate(
    question="What are the benefits of solar energy?",
    answer="Solar energy is renewable and reduces electricity bills. It has low maintenance costs.",
    context="Solar energy is a renewable source that lowers electricity costs and has low maintenance."
)
print(result.summary())

  LLMEVAL Evaluation Result
  Question : What are the benefits of solar energy?
  Answer   : Solar energy is renewable and reduces electricity bills. It has low maintenance 
------------------------------------------------------------
  bleu                 [######..............] 0.333
  rouge                [###########.........] 0.570
  token_overlap        [###############.....] 0.778
  keyword_coverage     [###############.....] 0.778
  answer_length        [####################] 1.000
  readability          [######..............] 0.310
------------------------------------------------------------
  Overall Score: 0.628  PASSED


In [3]:
# Individual local metrics
from llmevalkit import BLEUScore, ROUGEScore, KeywordCoverage, ReadabilityScore

answer = "Python is a high-level programming language used for web development and data science."
context = "Python is a high-level, interpreted programming language known for its simplicity."

for metric in [BLEUScore(), ROUGEScore(), KeywordCoverage(), ReadabilityScore()]:
    r = metric.evaluate(answer=answer, context=context)
    print("{:<22} {:.3f}".format(metric.name, r.score))

bleu                   0.316
rouge                  0.556
keyword_coverage       0.625
readability            0.476


## Part B: Compliance Testing (free, no API)

### PII Detection

In [4]:
from llmevalkit.compliance import PIIDetector

pii = PIIDetector()

# Test with PII
result = pii.evaluate(
    answer="Contact Mr. Raj Kumar at raj@gmail.com or call +91 98765 43210. His PAN is ABCDE1234F."
)
print("Score: {} (0.0 = PII found)".format(result.score))
print("PII count: {}".format(result.details["pii_count"]))
for item in result.details["pii_found"]:
    print("  Type: {:<15} Value: {}".format(item["type"], item["value"]))

Score: 0.0 (0.0 = PII found)
PII count: 7
  Type: email           Value: raj@gmail.com
  Type: phone_india     Value: +91 98765 43210
  Type: pan_india       Value: ABCDE1234F
  Type: upi_id          Value: raj@gmail
  Type: person_name     Value: Raj Kumar
  Type: date            Value: 98765 43210
  Type: organization    Value: PAN


In [5]:
# Test with clean text
result = pii.evaluate(
    answer="Solar energy reduces carbon emissions and electricity costs."
)
print("Score: {} (1.0 = no PII)".format(result.score))
print("Reason: {}".format(result.reason))

Score: 1.0 (1.0 = no PII)
Reason: No PII detected


### HIPAA Check (18 Safe Harbor Identifiers)

In [6]:
from llmevalkit.compliance import HIPAACheck

hipaa = HIPAACheck()

# Test with PHI
result = hipaa.evaluate(
    answer="Patient John Smith, SSN 123-45-6789, MRN: 12345678, email john@hospital.com, DOB 03/15/1980"
)
print("Score: {}".format(result.score))
print("HIPAA identifiers found: {}".format(result.details["identifiers_found"]))
print("Total violations: {}".format(result.details["total_violations"]))
for v in result.details["violations"]:
    print("  #{} {}: {}".format(v["identifier_number"], v["identifier_name"], v["value"]))

Score: 0.0
HIPAA identifiers found: [1, 3, 6, 7, 8]
Total violations: 6
  #6 Email addresses: john@hospital.com
  #7 Social Security numbers: 123-45-6789
  #3 Dates related to individual: 03/15/1980
  #8 Medical record numbers: MRN: 12345678
  #1 Names: John Smith
  #3 Dates related to individual: 12345678


In [7]:
# Test with safe medical text
result = hipaa.evaluate(
    answer="The patient recovery rate improved by 15% after the new treatment protocol."
)
print("Score: {} (1.0 = HIPAA compliant)".format(result.score))
print("Reason: {}".format(result.reason))

Score: 1.0 (1.0 = HIPAA compliant)
Reason: No HIPAA identifiers found. Output appears compliant with Safe Harbor.


### GDPR Check

In [8]:
from llmevalkit.compliance import GDPRCheck

gdpr = GDPRCheck()

# Test: user asks about data deletion, response ignores it
result = gdpr.evaluate(
    question="How do I delete my data?",
    answer="Thank you for your question. We store all user data securely in our servers."
)
print("Score: {:.3f}".format(result.score))
for issue in result.details["issues"]:
    print("  Article {}: {} - {}".format(issue["article"], issue["principle"], issue["description"]))

Score: 0.550
  Article 6-7: Consent - Discusses data processing without mentioning consent
  Article 17: Right to Erasure - User asked about data deletion but response does not acknowledge this right


In [9]:
# Test: proper response acknowledging erasure right
result = gdpr.evaluate(
    question="How do I delete my data?",
    answer="You can request erasure of your personal data by contacting our DPO. We will delete your data within 30 days."
)
print("Score: {:.3f}".format(result.score))
print("Reason: {}".format(result.reason))

Score: 0.700
Reason: No significant GDPR compliance issues detected


### India DPDP Act Check

In [10]:
from llmevalkit.compliance import DPDPCheck

dpdp = DPDPCheck()

# Test with Aadhaar and PAN exposure
result = dpdp.evaluate(
    answer="User Aadhaar: 1234 5678 9012, PAN: ABCDE1234F"
)
print("Score: {:.3f}".format(result.score))
for issue in result.details["issues"]:
    print("  Section {}: {}".format(issue["section"], issue["description"]))

Score: 0.400
  Section 8: India-specific PII exposed: aadhaar, pan_india
  Section 8: Personal data found in output (4 items)


In [11]:
# Test: children's data with targeted advertising
result = dpdp.evaluate(
    answer="We collect student data and share it with partners for targeted advertising."
)
print("Score: {:.3f}".format(result.score))
for issue in result.details["issues"]:
    print("  Section {}: {} - {}".format(issue["section"], issue["requirement"], issue["description"]))

Score: 0.250
  Section 4: Consent - Discusses data processing without mentioning consent
  Section 9: Children's Data Protection - Involves children's data without mentioning parental consent
  Section 9: Children's Data Protection - Behavioral monitoring or targeted advertising toward children


### EU AI Act Check

In [12]:
from llmevalkit.compliance import EUAIActCheck

eu = EUAIActCheck()

# Test: social scoring (prohibited)
result = eu.evaluate(
    answer="We calculate a social score for each citizen based on their online behavior."
)
print("Score: {}".format(result.score))
print("Risk level: {}".format(result.details["risk_level"]))

Score: 0.0
Risk level: unacceptable


In [13]:
# Test: high-risk medical context without human oversight
result = eu.evaluate(
    answer="Based on the medical diagnosis, the patient should take medication X immediately."
)
print("Score: {:.3f}".format(result.score))
print("Risk level: {}".format(result.details["risk_level"]))
print("High risk context: {}".format(result.details["high_risk_context"]))
for issue in result.details["issues"]:
    print("  Article {}: {}".format(issue["article"], issue["description"]))

Score: 0.700
Risk level: high
High risk context: medical diagnosis
  Article 50: High-risk AI context without AI disclosure
  Article 14: High-risk context (medical diagnosis) without mentioning human oversight


In [14]:
# Test: high-risk with proper human oversight mention
result = eu.evaluate(
    answer="The analysis suggests condition X. Please consult a doctor for professional advice and diagnosis."
)
print("Score: {:.3f}".format(result.score))
print("Risk level: {}".format(result.details["risk_level"]))

Score: 1.000
Risk level: limited


### Custom Compliance Rule

In [15]:
from llmevalkit.compliance import CustomRule

# Keyword-based rule (no API needed)
rule = CustomRule(
    rule="Output must not contain API keys or secrets",
    keywords=["api_key", "secret", "password", "token", "sk-"],
    use_llm=False,
)

result = rule.evaluate(answer="Set your api_key=sk-12345 in the config file")
print("Score: {}".format(result.score))
print("Reason: {}".format(result.reason))

Score: 0.0
Reason: Rule violated. Keywords found: api_key, sk-


## Part C: Combined Quality + Compliance

In [16]:
from llmevalkit import Evaluator

# Using compliance preset
evaluator = Evaluator(provider="none", preset="hipaa")
result = evaluator.evaluate(
    answer="Patient SSN: 123-45-6789, email john@hospital.com. Recovery rate: 85%."
)
print("Overall: {:.3f}".format(result.overall_score))
for name, m in result.metrics.items():
    print("  {:<22} {:.3f}".format(name, m.score))

Overall: 0.000
  pii_detector           0.000
  hipaa_check            0.000


In [17]:
# Mix quality metrics with compliance metrics
from llmevalkit import Evaluator, BLEUScore, ROUGEScore, KeywordCoverage
from llmevalkit.compliance import PIIDetector, HIPAACheck

evaluator = Evaluator(
    provider="none",
    metrics=[BLEUScore(), ROUGEScore(), KeywordCoverage(), PIIDetector(), HIPAACheck()],
)

result = evaluator.evaluate(
    answer="Solar energy reduces carbon emissions and lowers electricity bills.",
    context="Solar energy is a renewable source that reduces carbon emissions and electricity costs.",
)
print("Overall: {:.3f}".format(result.overall_score))
for name, m in result.metrics.items():
    print("  {:<22} {:.3f}  {}".format(name, m.score, m.reason[:60]))

Overall: 0.694
  bleu                   0.236  BLEU-4: 0.237 (BP=0.641)
  rouge                  0.566  ROUGE-1=0.636, ROUGE-2=0.400, ROUGE-L=0.636
  keyword_coverage       0.667  Covered 6/9 key terms (66.7%)
  pii_detector           1.000  No PII detected
  hipaa_check            1.000  No HIPAA identifiers found. Output appears compliant with Sa


## Part D: Batch Compliance Testing

In [19]:
from llmevalkit import Evaluator

evaluator = Evaluator(provider="none", preset="compliance_all")

cases = [
    {"question": "", "answer": "The treatment improved outcomes by 20%."},
    {"question": "", "answer": "Patient John, SSN 123-45-6789, was admitted on 03/15/1980."},
    {"question": "", "answer": "Contact support at help@company.com for assistance."},
    {"question": "", "answer": "We calculate a social score for each citizen."},
]

batch = evaluator.evaluate_batch(cases, show_progress=False)

for i, r in enumerate(batch.results):
    status = "PASS" if r.passed else "FAIL"
    print("Case {}: score={:.3f} {}".format(i + 1, r.overall_score, status))

print("\nPass rate: {:.0%}".format(batch.pass_rate))
print("Average: {:.3f}".format(batch.average_score))

Case 1: score=1.000 PASS
Case 2: score=0.480 FAIL
Case 3: score=0.420 FAIL
Case 4: score=0.800 PASS

Pass rate: 50%
Average: 0.675


## Part E: LLM-as-Judge (needs API key)

Set your Groq API key to run deeper compliance analysis.

In [20]:
import os
# Uncomment and set your key: put your groq api key
os.environ["GROQ_API_KEY"] = "gsk_"

In [21]:
import os

if os.getenv("GROQ_API_KEY"):
    from llmevalkit import Evaluator
    from llmevalkit.compliance import PIIDetector, HIPAACheck

    evaluator = Evaluator(
        provider="groq",
        model="llama-3.3-70b-versatile",
        metrics=[PIIDetector(use_llm=True), HIPAACheck(use_llm=True)],
    )
    result = evaluator.evaluate(
        answer="Mr. Kumar from Chennai was diagnosed with diabetes last March."
    )
    print("Score: {:.3f}".format(result.overall_score))
    for name, m in result.metrics.items():
        print("  {}: {:.3f}".format(name, m.score))
        if m.details.get("pii_found"):
            for p in m.details["pii_found"]:
                print("    Found: {} = {}".format(p["type"], p["value"]))
else:
    print("Set GROQ_API_KEY to run LLM-based compliance checks")
    print("LLM mode catches contextual PII that pattern matching misses")

Score: 0.000
  pii_detector: 0.000
    Found: person_name = Kumar
    Found: organization = Chennai
    Found: date = last March
    Found: person_name = Mr. Kumar
  hipaa_check: 0.000


## All Presets

| S.No. | Preset | What it checks |
|-------|--------|---------------|
| 1 | math / local | 7 local quality metrics (free) |
| 2 | rag | Faithfulness, Relevance, Hallucination (API) |
| 3 | chatbot | Relevance, Coherence, Toxicity (API) |
| 4 | pii | PII detection only |
| 5 | hipaa | PII + HIPAA 18 identifiers |
| 6 | gdpr | PII + GDPR principles |
| 7 | india / dpdp | PII + India DPDP Act |
| 8 | eu_ai | PII + GDPR + EU AI Act |
| 9 | compliance_all | All compliance metrics |
| 10 | rag_hipaa | RAG quality + HIPAA |
| 11 | rag_gdpr | RAG quality + GDPR |
| 12 | rag_india | RAG quality + India DPDP |

---

**pip install llmevalkit** | [GitHub](https://github.com/VK-Ant/llmevalkit) | Author: Venkatkumar Rajan